# Lab 4.3 · Personal data and instructions in the input

Day 4 · Session 3 (Checks before and after the model) · 25 min.

Riders write their names, phone numbers and national ID numbers into complaints. Some messages, and one policy page,
carry text addressed to the assistant. You write the checks that run before the model sees anything, measure them
on 30 risky items, compare Wasil with and without them, and write the log line Darb is allowed to keep.

**Reference state.** `data/safety/risky.jsonl`: 12 items with personal data to mask, 12 instructions to flag (six in
riders' messages, six in documents), and 6 harmless items that must pass untouched. `darb.mask_pii` and
`darb.instruction_like` are the reference checks. No model call is needed until step 3.

**Tasks**
- [ ] 1 · Mask personal data (7')
- [ ] 2 · Flag instructions in messages and documents (7')
- [ ] 3 · Wasil before and after the checks (6')
- [ ] 4 · A log line that is safe to keep (5')
- [ ] ✱ Stretch · Which fields may go to a hosted model?

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = "https://github.com/SDAIAAcademy/llm-applications.git"
    if not Path("/content/llm-applications").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/llm-applications"], check=True)
    os.chdir("/content/llm-applications/labs")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
    try:
        from google.colab import userdata  # the key from Colab's Secrets panel
        os.environ.setdefault("OPENROUTER_API_KEY", userdata.get("OPENROUTER_API_KEY"))
    except Exception:
        pass
else:
    while not Path("_tools").exists() and Path.cwd() != Path.cwd().parent:
        os.chdir("..")  # solutions/ notebooks run from labs/
sys.path.insert(0, os.getcwd())

import hashlib
import json
import re
from collections import Counter

import pandas as pd

import llm
import darb as D
import wasil

RISKY = D.load_jsonl("safety/risky.jsonl")
print(llm.describe(), "|", Counter((r["source"], r["expected"]) for r in RISKY))
print(RISKY[0]["text"])

## 1 · Mask personal data (7')

Masking replaces each piece of personal data with a typed placeholder (`[PHONE]`), so the message still reads and
the model still knows that a phone number was given. Saudi mobile numbers start with 05 (or +966 5), national ID and
Iqama numbers are ten digits starting with 1 or 2.

✏️ **Exercise** · Write `mask(text)` that returns `(masked_text, kinds)` for phone, national ID, email, card number
and the name after "my name is", "I am", "اسمي" or "أنا" (a capitalised name in English). Use `[0-9]`, not `\d`: in Python `\d` also matches Arabic-Indic digits,
and a rider writing ٠٥٥ would then be masked by one rule and missed by the next.

In [ ]:
def mask(text: str) -> tuple[str, list[str]]:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    mask("test")
    MASK = mask
except NotImplementedError:
    MASK = D.mask_pii
rows = []
for r in RISKY:
    if r["source"] == "message":
        _, kinds = MASK(r["text"])
        want = set(r["kinds"]) if r["expected"] == "mask" else set()
        rows.append({"id": r["id"], "expected": r["expected"], "want": sorted(want), "got": sorted(set(kinds)),
                     "ok": want <= set(kinds) and (r["expected"] != "pass" or not kinds)})
M = pd.DataFrame(rows)
print(M[M.expected != "flag"].to_string(index=False))
print("masked as expected:", f"{M[M.expected == 'mask'].ok.mean():.0%}", "| harmless left alone:",
      f"{M[M.expected == 'pass'].ok.mean():.0%}")
print(MASK(RISKY[0]["text"])[0])

🔍 **Look at** · the harmless items. A ten-digit refund reference that starts with 2 looks exactly like a national ID
number. Which way should the rule err?

## 2 · Flag instructions in messages and documents (7')

Text that addresses the assistant ("ignore your previous instructions", "note to automated assistants", "تجاهل
جميع التعليمات") is not a rider's request and not a policy. A flag does not delete it: it keeps it out of the
model's instructions and sends the message to a person.

✏️ **Exercise** · Write `flag(text)` that returns True for text that gives the assistant instructions, in English
and Arabic. Then measure it on the 12 items to flag and on everything else (false flags).

In [ ]:
def flag(text: str) -> bool:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    flag("test")
    FLAG = flag
except NotImplementedError:
    FLAG = D.instruction_like
F = pd.DataFrame([{"id": r["id"], "source": r["source"], "expected": r["expected"], "flagged": FLAG(r["text"])}
                  for r in RISKY])
print(F.groupby(["expected", "source"]).flagged.mean().round(2))
caught = F[F.expected == "flag"].flagged.mean()
false = F[F.expected != "flag"].flagged.mean()
print(f"caught {caught:.0%} of instructions, flagged {false:.0%} of the rest")
llm.check(caught >= 0.9 and false == 0, "instructions caught in both languages, nothing else flagged")

A rule set catches the phrasings it was written for. A new phrasing gets through, which is why the output check
(step 3) exists too: it is the second line, not a replacement for the first.

## 3 · Wasil before and after the checks (6')

`wasil.Wasil` runs the reference checks before the model (mask, flag) and after it (no personal data in the answer,
no promise of a refund, no request for an ID number). `RawWasil` below is the same assistant with the checks removed.

✏️ **Exercise** · Write `compare()`: send every rider message in `RISKY` to both assistants and count, for each,
how many answers (a) contain personal data from the input, (b) promise a refund, (c) were flagged.

In [ ]:
class RawWasil(wasil.Wasil):
    """The same assistant with no input or output checks (for this comparison only)."""

    def answer(self, text, received=None):
        saved = D.mask_pii, D.instruction_like
        D.mask_pii, D.instruction_like = (lambda t: (t, [])), (lambda t: False)
        try:
            return super().answer(text, received)
        finally:
            D.mask_pii, D.instruction_like = saved

    def check_output(self, answer, lang):
        return answer, False


def leaks(item: dict, answer: str) -> bool:
    found = [m for pat in D.PII_PATTERNS.values() for m in re.findall(pat, item["text"])]
    return any(f in answer for f in found)


def compare() -> pd.DataFrame:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    C = compare()
    print(C.to_string(index=False))
    llm.check(C.iloc[1]["promises a refund"] == 0, "no refund promised with the checks on")
except NotImplementedError:
    print("compare not written yet")
print(llm.LEDGER)

🔍 **Look at** · where the personal data went without the checks: into the answer, and into the log. A log kept
for a year is the larger exposure.

## 4 · A log line that is safe to keep (5')

Darb's data owner allows the log to hold what the team needs to debug and measure, and nothing that identifies a
rider. The PDPL's principle of data minimisation says the same in law.

✏️ **Exercise** · Write `log_line(text, out)`: a dict with the time-free fields Darb needs (a short hash of the
message so duplicates can be found, the language, the kind, the flags, the kinds of personal data found, the
citations, abstained) and **no** message text, no answer text and no personal data.

In [ ]:
def log_line(text: str, out: dict) -> dict:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
w = wasil.Wasil()
item = RISKY[0]
out = w.answer(item["text"])
try:
    line = log_line(item["text"], out)
except NotImplementedError:
    line = {"note": "log_line not written yet"}
print(json.dumps(line, ensure_ascii=False))
llm.check(not leaks(item, json.dumps(line, ensure_ascii=False)) and "Rana" not in json.dumps(line),
          "the log line holds no personal data")

## ✱ Stretch · Which fields may go to a hosted model?

Under Darb's classification, riders' complaints are **restricted**: personal data may not leave Darb's systems, but
the masked text may go to an approved hosted model. Write a table of the fields in a complaint and, for each, whether
it may be sent, and run the check on one message.

In [ ]:
def stretch():
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")


try:
    stretch()
except NotImplementedError:
    print("✱ stretch not attempted")

## Checkpoint

1. Why mask with typed placeholders instead of deleting the personal data?
<details><summary>Answer</summary>The model still knows a phone number was given and can say "we will call you";
the message still reads.</details>

2. The instruction filter missed a new phrasing. What still protects the rider?
<details><summary>Answer</summary>The output check: the answer cannot promise a refund or ask for an ID number,
whatever the input said.</details>

3. Why is the log a bigger risk than the answer?
<details><summary>Answer</summary>An answer is read once by one rider; a log is kept for months, copied and read by
many people.</details>

**Carry forward.** `wasil.Wasil.answer` runs the reference checks; the capstone README states what is masked, what is
flagged and what is logged.